# PSEi GRU: reproduce figures and tables
Run all cells in Google Colab. This rebuilds plots and numerical tables from
the saved PSEi runs on `codex/with-sop-2`; no model is trained. Chapter 1 is
illustrative, Chapter 3 contains retained diagrams, and Chapter 4 uses the
saved experiment data. Full training is available via `run_thesis.py`.


In [ ]:
from pathlib import Path
import os, subprocess, sys, tempfile
REPO = Path(tempfile.mkdtemp(prefix="psei-figures-")) / "stock-prediction"
subprocess.run(["git", "clone", "--depth", "1", "--branch", "codex/with-sop-2",
                "https://github.com/marianoluiz/stock-prediction.git", str(REPO)], check=True)
os.chdir(REPO)
print(subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


In [ ]:
# Colab already includes most of these. No Google Drive login is needed.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch", "pandas",
                "numpy", "matplotlib", "scipy", "scikit-learn", "yfinance"], check=True)


In [ ]:
subprocess.run([sys.executable, "run_thesis.py", "--figures-only"], check=True)


## Chapter 1: reproducible illustrations
Figures 1.2/1.3 use observed pre-test PSEi data and hypothetical forecasts. Figure 1.4 is the original synthetic fixed-cost example.


In [ ]:
from IPython.display import display, Image, Markdown
import pandas as pd
for number in range(1, 5):
    display(Markdown(f"### Figure 1.{number}"))
    display(Image(filename=str(REPO / f"results/chapter1/figure_1_{number}.png")))
display(pd.read_csv(REPO / "results/chapter1/loss_curve_values.csv").head())


## Chapter 3: retained source diagrams
These are static workflow and input-process-output diagrams.


In [ ]:
for number in (1, 2):
    display(Markdown(f"### Figure 3.{number}"))
    display(Image(filename=str(REPO / f"docs/figure_3_{number}.png")))


## Chapter 4: charts from saved test results
Means are fold means; the separate single-split worked-example curves are retained in `results/mse/` and `results/profit_aware/`.


In [ ]:
benchmark = REPO / "results/benchmark_summary_psei_20261025_203842.csv"
charts = benchmark.with_name(benchmark.stem + "_charts")
figures = {
    "4.1": charts / "per_fold_geo_diff.png",
    "4.2": charts / "strategy_comparison.png",
    "4.3": REPO / "results/sop2_lambda_sensitivity.png",
    "4.4": REPO / "results/significance.png",
    "4.5": REPO / "results/cost_comparison.png",
}
for number, path in figures.items():
    display(Markdown(f"### Figure {number}"))
    display(Image(filename=str(path)))


## Numerical tables
These are the numerical source tables, without Word formatting. The daily CSVs retain the observations behind the metrics.


In [ ]:
summary = pd.read_csv(benchmark)
display(Markdown("### Benchmark: per-fold results"))
display(summary)
display(Markdown("### Benchmark: means by GRU loss"))
metrics = ["cum_profit", "cum_profit_geo", "sharpe_like", "rmse", "directional_acc"]
display(summary.groupby("loss_type")[metrics].mean())
for title, filename in [
    ("Lambda comparison: fold means", "sop2_lambda_summary.csv"),
    ("Lambda comparison: individual folds", "sop2_lambda_comparison.csv"),
    ("Significance tests and confidence intervals", "significance.csv"),
    ("Cost comparison: paired conditions", "cost_comparison.csv"),
]:
    display(Markdown(f"### {title}"))
    display(pd.read_csv(REPO / "results" / filename))
display(Markdown("### Cost comparison summary"))
print((REPO / "results/cost_comparison_summary.txt").read_text())
